# Imports and load master dataset


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load master dataset
DATA_PATH = Path(r"C:\Users\Hp\water_leakage_ml\data\Net1_CMH")
master_df = pd.read_csv(DATA_PATH / "master_dataset.csv")

# Convert Timestamp to datetime
master_df["Timestamp"] = pd.to_datetime(master_df["Timestamp"])

print("Shape:", master_df.shape)
print("Columns:", master_df.columns.tolist())
print("\nFirst 3 rows:")
master_df.head(3)

Shape: (17520000, 30)
Columns: ['Index', 'Timestamp', 'Binary_Label', 'Pressure_Node_10', 'Pressure_Node_11', 'Pressure_Node_12', 'Pressure_Node_13', 'Pressure_Node_2', 'Pressure_Node_21', 'Pressure_Node_22', 'Pressure_Node_23', 'Pressure_Node_31', 'Pressure_Node_32', 'Pressure_Node_9', 'Flow_Link_10', 'Flow_Link_11', 'Flow_Link_110', 'Flow_Link_111', 'Flow_Link_112', 'Flow_Link_113', 'Flow_Link_12', 'Flow_Link_121', 'Flow_Link_122', 'Flow_Link_21', 'Flow_Link_22', 'Flow_Link_31', 'Flow_Link_9', 'Scenario', 'Diameter', 'Label']

First 3 rows:


,Index,Timestamp,Binary_Label,Pressure_Node_10,Pressure_Node_11,Pressure_Node_12,Pressure_Node_13,Pressure_Node_2,Pressure_Node_21,Pressure_Node_22,...,Flow_Link_12,Flow_Link_121,Flow_Link_122,Flow_Link_21,Flow_Link_22,Flow_Link_31,Flow_Link_9,Scenario,Diameter,Label
0,1,2017-01-01 00:00:00,0.0,92.944,89.658,82.317,83.756,36.576,84.905,83.847,...,14.4,25.2,3.6,54.0,18.0,7.2,406.8,Scenario-1,0.130441,0
1,2,2017-01-01 00:30:00,0.0,93.525,90.288,83.008,84.478,37.265,85.796,84.558,...,10.8,21.6,3.6,57.6,14.4,7.2,403.2,Scenario-1,0.130441,0
2,3,2017-01-01 01:00:00,0.0,94.188,91.007,83.731,85.228,37.985,86.611,85.311,...,7.2,21.6,0.0,57.6,14.4,7.2,399.6,Scenario-1,0.130441,0


# Define pressure and flow columns


In [2]:
# Identify pressure and flow columns automatically
pressure_cols = [c for c in master_df.columns if c.startswith("Pressure_")]
flow_cols = [c for c in master_df.columns if c.startswith("Flow_")]

print(f"Pressure columns ({len(pressure_cols)}): {pressure_cols}")
print(f"Flow columns ({len(flow_cols)}):     {flow_cols}")

Pressure columns (11): ['Pressure_Node_10', 'Pressure_Node_11', 'Pressure_Node_12', 'Pressure_Node_13', 'Pressure_Node_2', 'Pressure_Node_21', 'Pressure_Node_22', 'Pressure_Node_23', 'Pressure_Node_31', 'Pressure_Node_32', 'Pressure_Node_9']
Flow columns (13):     ['Flow_Link_10', 'Flow_Link_11', 'Flow_Link_110', 'Flow_Link_111', 'Flow_Link_112', 'Flow_Link_113', 'Flow_Link_12', 'Flow_Link_121', 'Flow_Link_122', 'Flow_Link_21', 'Flow_Link_22', 'Flow_Link_31', 'Flow_Link_9']


# Define rolling window size


In [3]:
# Each timestep = 30 minutes
# Window of 8 = 4 hours — captures short-term hydraulic behaviour
# Justification: leak evolution typically manifests within a few hours
WINDOW_SIZE = 8

print(f"Rolling window size: {WINDOW_SIZE} timesteps = {WINDOW_SIZE * 30} minutes = {WINDOW_SIZE * 30 / 60} hours")

Rolling window size: 8 timesteps = 240 minutes = 4.0 hours


# Compute mean pressure and mean flow (aggregate across sensors)


In [4]:
# Average across all sensor nodes per timestep
# This gives one representative pressure and flow reading per row
master_df["Mean_Pressure"] = master_df[pressure_cols].mean(axis=1)
master_df["Mean_Flow"] = master_df[flow_cols].mean(axis=1)

print("Mean_Pressure and Mean_Flow columns created")
print(master_df[["Timestamp", "Mean_Pressure", "Mean_Flow"]].head())

Mean_Pressure and Mean_Flow columns created
            Timestamp  Mean_Pressure  Mean_Flow
0 2017-01-01 00:00:00      72.878455  81.969231
1 2017-01-01 00:30:00      73.554091  79.200000
2 2017-01-01 01:00:00      74.245364  76.153846
3 2017-01-01 01:30:00      74.920636  75.323077
4 2017-01-01 02:00:00      75.647636  73.661538


# Pressure features (grouped by Scenario to avoid cross-scenario contamination)


In [5]:
print("Computing pressure features...")

master_df = master_df.sort_values(["Scenario", "Timestamp"]).reset_index(drop=True)

# Group by scenario so rolling stats don't bleed across scenario boundaries
grp = master_df.groupby("Scenario")["Mean_Pressure"]

# 1. Pressure Gradient (first difference)
master_df["Pressure_Gradient"] = grp.diff()

# 2. Rolling Mean Pressure
master_df["Rolling_Mean_Pressure"] = (
    grp.transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).mean())
)

# 3. Rolling Std Dev of Pressure
master_df["Rolling_Std_Pressure"] = (
    grp.transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).std())
)

print("Pressure gradient, rolling mean and rolling std dev created")

Computing pressure features...
Pressure gradient, rolling mean and rolling std dev created


# Pressure gradient anomaly indicator


In [6]:
print("Computing pressure anomaly indicator...")

# Compute threshold parameters from training data only
# For now we use the full dataset — we will recompute strictly on train set later
grad_mean = master_df["Pressure_Gradient"].mean()
grad_std  = master_df["Pressure_Gradient"].std()

PRESSURE_THRESHOLD = grad_mean - 3 * grad_std

print(f"Gradient mean:      {grad_mean:.4f}")
print(f"Gradient std:       {grad_std:.4f}")
print(f"Anomaly threshold:  {PRESSURE_THRESHOLD:.4f}")

# Binary indicator: 1 if gradient is below threshold (sudden pressure drop)
master_df["Pressure_Anomaly"] = (
    master_df["Pressure_Gradient"] < PRESSURE_THRESHOLD
).astype(int)

print(f"\nPressure anomaly flag distribution:")
print(master_df["Pressure_Anomaly"].value_counts())

Computing pressure anomaly indicator...
Gradient mean:      0.0018
Gradient std:       2.9076
Anomaly threshold:  -8.7209

Pressure anomaly flag distribution:
Pressure_Anomaly
0    17447093
1       72907
Name: count, dtype: int64


# Flow features


In [7]:
print("Computing flow features...")

grp_flow = master_df.groupby("Scenario")["Mean_Flow"]

# 1. Flow Difference (first difference)
master_df["Flow_Difference"] = grp_flow.diff()

# 2. Rolling Mean Flow
master_df["Rolling_Mean_Flow"] = (
    grp_flow.transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).mean())
)

# 3. Flow Variance
master_df["Flow_Variance"] = (
    grp_flow.transform(lambda x: x.rolling(window=WINDOW_SIZE, min_periods=1).var())
)

print("Flow difference, rolling mean and flow variance created")

Computing flow features...
Flow difference, rolling mean and flow variance created


# Flow spike indicator


In [8]:
print("Computing flow spike indicator...")

flow_diff_mean = master_df["Flow_Difference"].mean()
flow_diff_std  = master_df["Flow_Difference"].std()

FLOW_THRESHOLD = flow_diff_mean + 3 * flow_diff_std

print(f"Flow diff mean:    {flow_diff_mean:.4f}")
print(f"Flow diff std:     {flow_diff_std:.4f}")
print(f"Spike threshold:   {FLOW_THRESHOLD:.4f}")

# Binary indicator: 1 if flow difference exceeds threshold (sudden spike)
master_df["Flow_Spike"] = (
    master_df["Flow_Difference"] > FLOW_THRESHOLD
).astype(int)

print(f"\nFlow spike flag distribution:")
print(master_df["Flow_Spike"].value_counts())

Computing flow spike indicator...
Flow diff mean:    -0.0018
Flow diff std:     9.0062
Spike threshold:   27.0169

Flow spike flag distribution:
Flow_Spike
0    17459939
1       60061
Name: count, dtype: int64


# Temporal features


In [9]:
print("Computing temporal features...")

master_df["Hour_Of_Day"] = master_df["Timestamp"].dt.hour
master_df["Day_Of_Week"] = master_df["Timestamp"].dt.dayofweek

print("Hour_Of_Day and Day_Of_Week created")
print(master_df[["Timestamp", "Hour_Of_Day", "Day_Of_Week"]].head(8))

Computing temporal features...
Hour_Of_Day and Day_Of_Week created
            Timestamp  Hour_Of_Day  Day_Of_Week
0 2017-01-01 00:00:00            0            6
1 2017-01-01 00:30:00            0            6
2 2017-01-01 01:00:00            1            6
3 2017-01-01 01:30:00            1            6
4 2017-01-01 02:00:00            2            6
5 2017-01-01 02:30:00            2            6
6 2017-01-01 03:00:00            3            6
7 2017-01-01 03:30:00            3            6


# Drop NaN rows created by diff and rolling


In [10]:
before = len(master_df)
master_df = master_df.dropna(subset=[
    "Pressure_Gradient",
    "Rolling_Mean_Pressure",
    "Rolling_Std_Pressure",
    "Flow_Difference",
    "Rolling_Mean_Flow",
    "Flow_Variance"
])
after = len(master_df)

print(f"Rows before dropna: {before:,}")
print(f"Rows after dropna:  {after:,}")
print(f"Rows dropped:       {before - after:,}")

Rows before dropna: 17,520,000
Rows after dropna:  17,519,000
Rows dropped:       1,000


# Final feature set and save


In [11]:
# Define the final feature columns for the model
feature_cols = [
    "Pressure_Gradient",
    "Rolling_Mean_Pressure",
    "Rolling_Std_Pressure",
    "Pressure_Anomaly",
    "Flow_Difference",
    "Rolling_Mean_Flow",
    "Flow_Variance",
    "Flow_Spike",
    "Hour_Of_Day",
    "Day_Of_Week"
]

# Keep only what the model needs
model_df = master_df[feature_cols + ["Label"]].copy()

print("Final dataset shape:", model_df.shape)
print("\nClass distribution after feature engineering:")
print(model_df["Label"].value_counts().sort_index())
print("\nFeature preview:")
model_df.head()

Final dataset shape: (17519000, 11)

Class distribution after feature engineering:
Label
0    13710752
1     1900959
2     1907289
Name: count, dtype: int64

Feature preview:


,Pressure_Gradient,Rolling_Mean_Pressure,Rolling_Std_Pressure,Pressure_Anomaly,Flow_Difference,Rolling_Mean_Flow,Flow_Variance,Flow_Spike,Hour_Of_Day,Day_Of_Week,Label
1,0.675636,73.216273,0.477747,0,-2.769231,80.584615,3.834320,0,0,6,0
2,0.691273,73.559303,0.683469,0,-3.046154,79.107692,8.461065,0,1,6,0
3,0.675273,73.899636,0.880186,0,-0.830769,78.161538,9.221538,0,1,6,0
4,0.727000,74.249236,1.091855,0,-1.661538,77.261538,10.966154,0,2,6,0
5,0.705273,74.599848,1.300496,0,-1.384615,76.430769,12.913988,0,2,6,0


# Save engineered dataset


In [12]:
model_df.to_csv(DATA_PATH / "featured_dataset.csv", index=False)
print("featured_dataset.csv saved successfully")

featured_dataset.csv saved successfully
